# Sorghum detect-out coordinates

Convert the July 15, 2021 detection centers from WGS84 longitude/latitude to local UTM meters. The fixed origin is shared across plots and can be reused for other dates. X is east and Y is north; EvoEngine places these as X and Z.

Install `pandas`, `numpy`, `matplotlib`, and `pyproj` in the notebook kernel. Set `SOURCE`, `OUTPUT_DIR`, `PLOTS`, and `MIN_CONFIDENCE` in the next cell. The exported `*_xy.txt` files use the count-then-X/Y format expected by `SorghumCoordinates::ImportFromFile`. The text files apply `MIN_CONFIDENCE` (72 of 153 target detections at the default 0.5); the CSV preserves every detection and its confidence. Coordinates are detection candidates; this notebook does not deduplicate or verify plants. Set the SorghumCoordinates sample X/Y ranges to include the exported points and keep its distance factor at 1.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyproj import Transformer

SOURCE = Path(r"D:\SorghumFieldsData\2021-07-15__09-45-31-924_sorghum_detect_out\detect_out\2021-07-15__09-45-31-924_sorghum_detection.csv")
OUTPUT_DIR = Path("out/sorghum-detect-2021-07-15")
PLOTS = (7403, 7404, 7407, 7408, 7411, 7412, 7415, 7416)
MIN_CONFIDENCE = 0.5
ORIGIN_LON_LAT = (-111.975, 33.0773)  # Fixed near the target plots; WGS84 degrees.

transformer = Transformer.from_crs("EPSG:4326", "EPSG:32612", always_xy=True)
origin_east, origin_north = transformer.transform(*ORIGIN_LON_LAT)
detections = pd.read_csv(SOURCE)
detections = detections.loc[detections["plot"].isin(PLOTS)].copy()
if detections.empty:
    raise ValueError("No detections found for the selected plots")
east, north = transformer.transform(detections["lon"].to_numpy(), detections["lat"].to_numpy())
detections["utm_east_m"] = east
detections["utm_north_m"] = north
detections["X_m"] = east - origin_east
detections["Y_m"] = north - origin_north
print(f"{len(detections)} detections in {len(PLOTS)} plots; fixed origin {ORIGIN_LON_LAT}, EPSG:32612")


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
detections.to_csv(OUTPUT_DIR / "detections_projected.csv", index=False)
selected = detections.loc[detections["pred_conf"] >= MIN_CONFIDENCE].copy()

def write_xy(path, frame):
    with path.open("w", newline="\n") as file:
        file.write(f"{len(frame)}\n")
        np.savetxt(file, frame[["X_m", "Y_m"]].to_numpy(), fmt="%.8f %.8f")

write_xy(OUTPUT_DIR / "selected_plots_xy.txt", selected)
for plot in PLOTS:
    subset = selected.loc[selected["plot"] == plot]
    write_xy(OUTPUT_DIR / f"plot_{plot}_xy.txt", subset)
    print(f"plot {plot}: {len(subset)} of {sum(detections['plot'] == plot)} detections exported")
print("Output:", OUTPUT_DIR.resolve())


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for plot in PLOTS:
    subset = selected.loc[selected["plot"] == plot]
    ax.scatter(subset["X_m"], subset["Y_m"], s=18, label=str(plot))
ax.set(xlabel="East from origin (m)", ylabel="North from origin (m)",
       title=f"Sorghum detections, confidence ≥ {MIN_CONFIDENCE:g}")
ax.set_aspect("equal", adjustable="box")
ax.grid(alpha=0.3)
ax.legend(title="Plot", ncol=2, loc="upper left", bbox_to_anchor=(1.02, 1))
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "selected_plots_xy.png", dpi=180)
plt.show()
